# RedditSIEM – Extremist Comment Classifier
### Demo su Google Colab

Pipeline: **entity detection → ABSA ensemble (DeBERTa + pyabsa) → detoxify → score**

> **Nota:** la prima esecuzione scarica i modelli (~2-4 GB). Usa **Runtime → Change runtime type → T4 GPU** per velocizzare l'inferenza.

## 1 · Installazione dipendenze

In [ ]:
%%capture
# Installa tutte le dipendenze
!pip install torch transformers detoxify nltk scipy
# pyabsa opzionale: se fallisce il classificatore usa solo DeBERTa
!pip install pyabsa || true

In [ ]:
import os, sys

REPO = "RedditSIEM"
BRANCH = "claude/extremist-comment-classifier-bQnxN"

if not os.path.exists(REPO):
    # Prima esecuzione: clona il repo
    # Se il repo è privato sostituisci con:
    # !git clone https://TUO_TOKEN@github.com/5iraFic/RedditSIEM.git
    !git clone https://github.com/5iraFic/RedditSIEM.git
    os.chdir(REPO)
    !git checkout {BRANCH}
else:
    # Esecuzioni successive: aggiorna il codice
    os.chdir(REPO)
    !git pull origin {BRANCH}

# Aggiungi la directory al path di Python
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

!git log --oneline -3

## 2 · Caricamento del classificatore

I modelli vengono scaricati automaticamente alla prima chiamata:
- `yangheng/deberta-v3-base-absa-v1.1` (~600 MB)
- `pyabsa` ATEPC multilingual checkpoint (~300 MB)
- `detoxify` multilingual XLM-RoBERTa (~1.1 GB)

In [ ]:
import logging

# Mostra solo WARNING e sopra (nasconde i LOAD REPORT verbosi dei modelli)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
# Silenzia completamente i warning ripetuti di pyabsa
logging.getLogger("extremist_classifier.absa").setLevel(logging.ERROR)

from extremist_classifier import ExtremistCommentClassifier

clf = ExtremistCommentClassifier(
    use_deberta     = True,
    use_pyabsa      = True,     # tenta pyabsa; se non disponibile usa solo DeBERTa
    deberta_weight  = 0.6,
    pyabsa_weight   = 0.4,
    absa_weight     = 0.6,
    toxicity_weight = 0.4,
    detoxify_model  = "multilingual",
    device          = "auto",
)
print("✓ Classificatore pronto")

## 3 · Classifica un singolo commento

In [ ]:
comment = (
    "We must support the mujahideen in their jihad against the kuffar. "
    "May Allah grant them victory and destroy the infidels!"
)

result = clf.classify(
    comment    = comment,
    comment_id = "t1_demo001",
    subreddit  = "r/example",
    author     = "u/testuser",
)

print(result.summary())

In [ ]:
import json
# Output strutturato pronto per SIEM (Elasticsearch / Splunk)
print(json.dumps(result.to_siem_event(), indent=2))

## 4 · Batch evaluation su commenti di test

In [ ]:
SAMPLES = [
    # (testo, rischio_atteso)
    ("We must support the mujahideen in their jihad against the kuffar.",         "HIGH"),
    ("ISIS is right to establish the caliphate. Death to Jews and infidels.",      "HIGH"),
    ("The shaheed who carried out the operation is a true martyr.",               "HIGH"),
    ("People say ISIS is brutal, but remember what Americans did in Iraq.",        "LOW"),
    ("ISIS must be stopped. Their attacks on civilians are unacceptable.",         "NONE"),  # counter-signal
    ("I visited the mosque last Friday. The sermon was about patience.",           "NONE"),
    ("Just made some amazing shawarma for dinner. Highly recommend!",              "NONE"),
]

batch = [{"body": t, "id": f"t1_{i:03d}"} for i, (t, _) in enumerate(SAMPLES)]
results = clf.classify_batch(batch)

print(f"{'✓/✗':<4} {'Label':>8}  {'Expected':>8}  {'Score':>6}  Entità trovate")
print("-" * 65)
for (text, expected), res in zip(SAMPLES, results):
    ok = "✓" if res.risk_label == expected else "✗"
    entities = ", ".join(res.entity_labels_found) or "—"
    print(f"{ok:<4} {res.risk_label:>8}  {expected:>8}  {res.doc_score:>6.3f}  {entities}")

## 5 · Classifica un tuo commento personalizzato

In [ ]:
# ← Modifica questo testo con il commento che vuoi testare
MY_COMMENT = "Insert your Reddit comment here..."

r = clf.classify(MY_COMMENT)
print(r.summary())
print()

# Dettaglio per ogni entità trovata
for s in r.sentences:
    print(f"  Entità : {s.entity_label!r} ({s.entity_type})")
    print(f"  Frase  : {s.sentence!r}")
    print(f"  ABSA   : {s.absa_sentiment} (conf={s.absa_confidence:.3f}, agreement={s.absa_agreement})")
    print(f"  Tox    : toxicity={s.toxicity_scores.get('toxicity',0):.3f}  "
          f"identity_attack={s.toxicity_scores.get('identity_attack',0):.3f}  "
          f"threat={s.toxicity_scores.get('threat',0):.3f}")
    print(f"  Signal : {s.signal_type}  score={s.sentence_extremism_score:.3f}")
    print()

## 6 · (Opzionale) Solo DeBERTa, senza pyabsa

Se pyabsa è lento o dà problemi, puoi disabilitarlo e usare solo DeBERTa:

In [ ]:
clf_lite = ExtremistCommentClassifier(
    use_deberta    = True,
    use_pyabsa     = False,   # solo DeBERTa
    detoxify_model = "multilingual",
)

r = clf_lite.classify("The mujahideen are heroes fighting the kuffar.")
print(r.summary())